## Opponent squad ensembles: C-vine copula and swap sampler

Builds $\mathbb{W}_o$, the plausible squads of one opponent for a gameweek, from their
personalized selection model (`longitudinal.ipynb`'s `fit_manager_pipeline` /
`alpha_manager`), in two ways that can be compared:

* a sparse C-vine copula over their positional marginals (`squad_sampler="vine"`), and
* their current squad plus keep/buy swaps (`squad_sampler="swap"`, eq. 3.6).

On this data the copula's fitted dependence reduces to independence (diagnostics recorded
with every vine result), which is what motivates the swap sampler.

Bivariate copula fitting, AIC-based family selection, and vine simulation use
`pyvinecopulib` rather than a from-scratch implementation.

One correction to the write-up: the stated boundary formulas
$a_{t,k}=(1-X_{t,k})(1-p_{t,k})$, $b_{t,k}=X_{t,k}+(1-X_{t,k})(1-p_{t,k})$ collapse to
$a=b=1-p_{t,k}$ when $X_{t,k}=0$ (a zero-width interval, not the stated
$[0, 1-p_{t,k})$). The cases actually given for $u_{t,k}$'s support --
$[0, 1-p_{t,k})$ when not selected, $[1-p_{t,k}, 1]$ when selected -- are unambiguous and
are exactly the standard _distributional transform_ used to continuize a discrete
(here Bernoulli($p_{t,k}$)) margin for copula modelling, so that's what's implemented
below: $a_{t,k} = X_{t,k}\cdot(1-p_{t,k})$, $b_{t,k} = X_{t,k} + (1-X_{t,k})(1-p_{t,k})$.


### Setup

Loads the player data, the league fits (for `feats`/`X_mean`/`X_std`/`beta_mean`) and the
selection-model helpers. Doesn't re-run any PyMC fitting: it reads the
`manager_fits_{GW}.joblib` files produced by `longitudinal.ipynb`'s per-round loop.


In [20]:
import ast

import numpy as np
import pandas as pd
import joblib
import pyvinecopulib as pv
from scipy.optimize import minimize_scalar
from scipy.special import expit, logsumexp
from scipy.stats import kendalltau
from sklearn.linear_model import LogisticRegression

player_data = pd.read_csv("../../inputs/rolled_data_24-25_xp_fixed.csv")
league_selections = pd.read_csv("../../inputs/league_selections_df.csv")

LEAGUE_FITS = {
    "Goalkeeper": joblib.load("../league betas/league_models_gk"),
    "Defender": joblib.load("../league betas/league_models_def"),
    "Midfielder": joblib.load("../league betas/league_models_mid"),
    "Forward": joblib.load("../league betas/league_models_fwd"),
}

POSITION_BINARY_COL = {
    "Goalkeeper": "goalkeeper_binary",
    "Defender": "defender_binary",
    "Midfielder": "midfielder_binary",
    "Forward": "forward_binary",
}

SQUAD_SLOTS = {"Goalkeeper": 2, "Defender": 5, "Midfielder": 5, "Forward": 3}


def get_league_fit_for_round(league_fits, target_round):
    """Pick the league fit at `target_round`, or the most recent one available
    before it (see longitudinal.ipynb for the full rationale)."""
    available = [gw for gw in league_fits if gw <= target_round]
    if not available:
        raise ValueError(f"No league fit available at or before round {target_round}")
    return league_fits[max(available)]


def _position_pool(player_data, position, rnd, feats):
    """The player pool for `position` at gameweek `rnd`, sorted by element id
    ascending -- matches the *_binary columns in league_selections for that round."""
    pool = player_data[
        (player_data["round"] == rnd) & (player_data["position"] == position)
    ].sort_values("element")
    elements = pool["element"].to_numpy()
    X_raw = pool[feats].fillna(0).to_numpy(dtype=float)
    return elements, X_raw


def alpha_manager(manager_fit, league_fit, X_new_raw, y_prev):
    """alpha_{j,m,t,k} (eq:alpha_m) for a fitted manager model at new raw features
    X_new_raw and retention lag y_prev."""
    X_std_new = (X_new_raw - league_fit["X_mean"]) / league_fit["X_std"]
    beta_m = league_fit["beta_mean"] + manager_fit["delta_beta_mean"]
    linear = (
        manager_fit.get("intercept_mean", 0.0)
        + X_std_new @ beta_m
        + manager_fit["gamma_mean"] * y_prev
    )
    linear = np.clip(linear, -30, 30)
    return np.exp(linear)


def marginal_inclusion_probabilities(weights, k, rng, n_sim=3000):
    """
    Efraimidis-Spirakis (A-ES) weighted sampling without replacement: draw
    key_i = U_i^(1/w_i) for U_i~Uniform(0,1), take the k items with the largest keys.
    This reproduces exactly sequential (Plackett-Luce) weighted sampling without
    replacement, and by linearity of expectation the resulting per-item inclusion
    probabilities always sum to exactly k -- turning `weights` (alpha_manager's raw,
    arbitrarily-scaled output) into genuine marginal "P(this player is one of the k
    picked)" probabilities. See the markdown above the pool-construction cell for why
    this replaces the naive alpha/alpha.sum() (which sums to 1 per position, not k).
    """
    weights = np.clip(np.asarray(weights, dtype=float), 1e-12, None)
    n = len(weights)
    U = rng.uniform(size=(n_sim, n))
    keys = U ** (1.0 / weights)[None, :]
    topk_idx = np.argpartition(-keys, kth=k - 1, axis=1)[:, :k]
    counts = np.bincount(topk_idx.ravel(), minlength=n)
    return counts / n_sim


def detect_squad_overhaul(team_id, gw, league_selections, threshold=8):
    """
    Flags gameweek `gw` as a likely Wildcard/Free Hit week for `team_id` (see
    longitudinal.ipynb): more than `threshold` of the 15 squad slots changed from
    gw-1's actual squad is the signature of a chip-driven full rebuild, which
    alpha_manager has no signal for. Returns (n_changed, is_overhaul); both None if
    gw-1 or gw isn't in league_selections for this team.
    """
    prev_rows = league_selections[
        (league_selections["team_id"] == team_id)
        & (league_selections["round"] == gw - 2)
    ]
    cur_rows = league_selections[
        (league_selections["team_id"] == team_id)
        & (league_selections["round"] == gw - 1)
    ]
    if len(prev_rows) == 0 or len(cur_rows) == 0:
        return None, None
    prev_squad = set(ast.literal_eval(prev_rows.iloc[0]["squad"]))
    cur_squad = set(ast.literal_eval(cur_rows.iloc[0]["squad"]))
    n_changed = len(cur_squad - prev_squad)
    return n_changed, n_changed > threshold


def compute_free_transfers_available(
    team_id, target_round, league_selections, cap=5, overhaul_threshold=8
):
    """
    Simulates `team_id`'s banked free transfers (FTs) entering `target_round`, from
    FPL's accrual rule: 1 new FT per week, unused ones roll over up to `cap` banked at
    once; making more transfers than banked forces a points hit and drains the bank to
    0 (so next week starts back at the baseline 1). GW1 is initial squad formation (no
    transfer concept), so the bank starts at 1 entering GW2.

    Wildcard/Free Hit weeks (flagged by detect_squad_overhaul) are transparent to the
    bank: neither consumes nor resets it, so that week's actual squad-composition
    change is ignored here and the bank just accrues its usual +1 (capped).

    This uses only observed squad snapshots (league_selections), so it can look back
    to GW1 regardless of `min_round` elsewhere in the pipeline (model features aren't
    needed here).
    """
    ft_bank = 1
    for w in range(2, target_round):
        n_changed, is_overhaul = detect_squad_overhaul(
            team_id, w, league_selections, threshold=overhaul_threshold
        )
        if n_changed is None:
            continue
        if is_overhaul:
            ft_bank = min(cap, ft_bank + 1)
        elif n_changed <= ft_bank:
            ft_bank = min(cap, (ft_bank - n_changed) + 1)
        else:
            ft_bank = 1
    return ft_bank

In [21]:
# The gameweek and opponent for the worked example below; manager_fits_{GW}.joblib
# (longitudinal.ipynb) holds the fit for that week's opponent.
GW = 38
TEAM_ID = 13603


Loaded manager_fits for GW38, team 13603: positions ['Goalkeeper', 'Defender', 'Midfielder', 'Forward']


### Selection marginals

$p_{k,t}$, the probability player $k$ fills one of a position's squad slots in week $t$, comes in
two forms:

* **Efraimidis-Spirakis marginals** (`marginal_inclusion_probabilities`): $\alpha$ from
  `alpha_manager` used as weights for drawing exactly the position's slot count without
  replacement, so the probabilities sum to the slot count rather than to 1.
* **Keep/buy marginals (eq. 3.6)** (`keep_buy_marginals`, with a `calib` from
  `fit_keep_buy_calibration`): each player owned last week is kept with probability
  $\text{logit}^{-1}(a_j + b_j\eta_{k,t})$, and the expected number of buys is spread over
  everyone else by $\text{softmax}(\tau_j\eta)$, with
  $\eta_{k,t} = (\beta^{league}+\Delta\beta_m)X_{t,k}$. $(a_j, b_j, \tau_j)$ are fit by
  maximum likelihood on the whole league's history before the target gameweek, on the
  league-level $\eta$, then applied to the opponent's own $\eta$. A single $\alpha$ can't
  capture both "keep ~93% of the squad" and "pick the right newcomers", which the keep/buy
  split does; the marginals still sum to the slot count.

`y_prev` (last week's ownership) is read against last week's own player pool, which differs
in size from this week's most weeks.


In [22]:
def get_owned(team_id, position, rnd, elements):
    """Ownership map {element: 0/1} for `position` at round `rnd`, or None if that
    round's selection export doesn't line up with `elements`."""
    # print(rnd)
    rows = league_selections[
        (league_selections["team_id"] == team_id) & (league_selections["round"] == rnd)
    ]
    if len(rows) == 0:
        return None
    y = np.asarray(
        ast.literal_eval(rows.iloc[0][POSITION_BINARY_COL[position]]), dtype=float
    )
    if len(y) != len(elements):
        return None
    return dict(zip(elements.tolist(), y.tolist()))


def position_alpha_at_week(team_id, position, manager_fit, league_fit, rnd):
    """Raw alpha_{j,m,rnd,k} (eq:alpha_m) for every player in `position`'s pool at
    week `rnd`, plus that week's ownership map (or None if unavailable). Used both as
    the weights for marginal_inclusion_probabilities and, later, as the weights for
    keying the vine's correlated draws when constructing full candidate squads."""
    elements, X_raw = _position_pool(
        player_data, position, rnd - 1, league_fit["feats"]
    )
    # recompute the previous week's pool fresh -- it's a different size than `elements`
    # most weeks, so reusing `elements` here would silently zero out y_prev (see markdown)
    prev_elements, _ = _position_pool(
        player_data, position, rnd - 2, league_fit["feats"]
    )
    owned_prev_map = get_owned(team_id, position, rnd - 2, prev_elements) or {}
    y_prev = np.array([owned_prev_map.get(e, 0.0) for e in elements])
    alpha = alpha_manager(manager_fit, league_fit, X_raw, y_prev)
    owned_map = get_owned(team_id, position, rnd - 1, elements)

    # print(f'Pos Margi at Week {rnd} {position} --->  {owned_map}')
    return dict(zip(elements.tolist(), alpha.tolist())), owned_map


def position_week_design(team_id, position, league_fit, rnd):
    """Same week alignment as position_alpha_at_week: the pool and ownership at
    round rnd-1, the retention lag from round rnd-2 (its own pool, see markdown above),
    and the features standardized on `league_fit`'s training mean/std."""
    elements, X_raw = _position_pool(player_data, position, rnd - 1, league_fit["feats"])
    prev_elements, _ = _position_pool(
        player_data, position, rnd - 2, league_fit["feats"]
    )
    owned_prev_map = get_owned(team_id, position, rnd - 2, prev_elements) or {}
    y_prev = np.array([owned_prev_map.get(e, 0.0) for e in elements])
    owned_map = get_owned(team_id, position, rnd - 1, elements)
    X_std = (X_raw - league_fit["X_mean"]) / league_fit["X_std"]
    return elements, X_std, y_prev, owned_map


def keep_buy_marginals(eta, y_prev, k, calib):
    """eq. 3.6: P(kept) = logit^-1(a + b*eta) for last week's squad; the expected
    number of buys (k - sum of keep probabilities) is spread over everyone else by
    softmax(tau * eta). Sums to k (up to the cap at 1)."""
    p = np.zeros(len(eta))
    owned = y_prev == 1
    p_keep = expit(calib["a"] + calib["b"] * eta[owned])
    p[owned] = p_keep
    if (~owned).any():
        n_buy = max(k - p_keep.sum(), 0.0)
        z = calib["tau"] * eta[~owned]
        p[~owned] = np.minimum(n_buy * np.exp(z - logsumexp(z)), 1.0)
    return p


def fit_keep_buy_calibration(position, league_fit, max_round, min_round=5):
    """
    Maximum-likelihood (a, b, tau) for keep_buy_marginals, pooled over every league
    manager's non-overhaul weeks rnd in [min_round, max_round) -- pass the same cutoff
    as the history window so nothing from the target week leaks in. Uses the
    league-level eta = beta^league . X (see markdown above). min_round=5 keeps round
    rnd-1 inside the league fits' training window (rounds > 3). Returns None when there
    is no such week yet (the first gameweeks of the season); tau is 1 until a buy has
    been seen in the position.
    """
    keep_eta, keep_x, buy_groups = [], [], []
    for team_id in league_selections["team_id"].unique():
        for rnd in range(min_round, max_round):
            if detect_squad_overhaul(team_id, rnd, league_selections)[1]:
                continue
            elements, X_std, y_prev, owned_map = position_week_design(
                team_id, position, league_fit, rnd
            )
            if owned_map is None or y_prev.sum() == 0:
                continue
            x = np.array([owned_map[e] for e in elements])
            eta = X_std @ league_fit["beta_mean"]
            owned = y_prev == 1
            keep_eta.append(eta[owned])
            keep_x.append(x[owned])
            if x[~owned].sum() > 0:
                buy_groups.append((eta[~owned], x[~owned]))

    if not keep_eta:
        # too early in the season: no earlier week to learn keeping from
        return None
    keep_eta, keep_x = np.concatenate(keep_eta), np.concatenate(keep_x)
    if 0 < keep_x.mean() < 1:
        keep_model = LogisticRegression(C=1.0).fit(keep_eta[:, None], keep_x)
        a, b = float(keep_model.intercept_[0]), float(keep_model.coef_[0, 0])
    else:  # nobody ever sold (or kept): constant keep probability
        rate = np.clip(keep_x.mean(), 1e-3, 1 - 1e-3)
        a, b = float(np.log(rate / (1 - rate))), 0.0

    def buy_nll(tau):
        # conditional logit: each observed buy is a softmax(tau * eta) draw over the
        # players not owned last week
        return -sum(
            (x * tau * eta).sum() - x.sum() * logsumexp(tau * eta)
            for eta, x in buy_groups
        )

    if buy_groups:
        tau = minimize_scalar(buy_nll, bounds=(0.01, 50.0), method="bounded").x
    else:
        # no buys seen yet in this position (rare early on, e.g. goalkeepers): leave
        # eta unsharpened rather than dropping the whole calibration
        tau = 1.0
    return {
        "a": a,
        "b": b,
        "tau": float(tau),
        "n_keep_obs": len(keep_x),
        "n_buy_weeks": len(buy_groups),
    }


def position_marginals_at_week(
    team_id, position, manager_fit, league_fit, rnd, rng, calib=None
):
    """p_{k,rnd}, the marginal probability player k is one of position's squad slots
    that week, plus that week's ownership map (or None if unavailable). With `calib`
    (from fit_keep_buy_calibration) these are the keep/buy marginals of eq. 3.6;
    without it, position_alpha_at_week's weights rescaled to selection probabilities
    via marginal_inclusion_probabilities (see markdown above)."""
    if calib is not None:
        elements, X_std, y_prev, owned_map = position_week_design(
            team_id, position, league_fit, rnd
        )
        eta = X_std @ (league_fit["beta_mean"] + manager_fit["delta_beta_mean"])
        p = keep_buy_marginals(eta, y_prev, SQUAD_SLOTS[position], calib)
        return dict(zip(elements.tolist(), p.tolist())), owned_map

    alpha_map, owned_map = position_alpha_at_week(
        team_id, position, manager_fit, league_fit, rnd
    )
    elements = np.array(list(alpha_map.keys()))
    alpha = np.array(list(alpha_map.values()))
    p = marginal_inclusion_probabilities(alpha, SQUAD_SLOTS[position], rng)
    return dict(zip(elements.tolist(), p.tolist())), owned_map


Pool M size: 121  (by position: {'Defender': 31, 'Forward': 30, 'Goalkeeper': 30, 'Midfielder': 30})


## Running across every gameweek we have fits for

`longitudinal.ipynb`'s per-round loop produced `manager_fits_{GW}.joblib` for every `GW` in 6-38,
each fit for that round's opponent under the round-robin fixture (manager 205, seeds 61/16,
reproduced here so this notebook doesn't need to re-import `longitudinal.ipynb`).


In [29]:
import random


def create_fixtures(managers, seed=None):
    """Randomized single round-robin fixture (verbatim from longitudinal.ipynb)."""
    if seed is not None:
        random.seed(seed)
    managers = managers.copy()
    random.shuffle(managers)
    fixed = managers[0]
    rotating = managers[1:]
    fixtures = {}
    for gw in range(1, len(managers)):
        current = [fixed] + rotating
        gw_fixtures = {}
        for i in range(len(current) // 2):
            m1, m2 = current[i], current[-(i + 1)]
            gw_fixtures[m1] = m2
            gw_fixtures[m2] = m1
        fixtures[gw] = gw_fixtures
        rotating = [rotating[-1]] + rotating[:-1]
    return fixtures


def get_opponent(manager_id, gameweek, fixtures):
    return fixtures[gameweek][manager_id]


managers = [
    205,
    901,
    4213,
    7497,
    13603,
    23881,
    38054,
    38351,
    69718,
    152245,
    198971,
    577273,
    593532,
    696127,
    749244,
    1222335,
    1441046,
    1546929,
    2634457,
    4131447,
]
fixtures_leg_1 = create_fixtures(managers, seed=61)
fixtures_leg_2 = create_fixtures(managers, seed=16)
opponents_by_round = {
    gw: get_opponent(205, gw, fixtures_leg_1) for gw in range(1, 20)
} | {gw + 19: get_opponent(205, gw, fixtures_leg_2) for gw in range(1, 20)}
print({gw: opponents_by_round[gw] for gw in range(6, 39)})

{6: 23881, 7: 38054, 8: 13603, 9: 593532, 10: 749244, 11: 696127, 12: 4131447, 13: 152245, 14: 1441046, 15: 1222335, 16: 1546929, 17: 198971, 18: 2634457, 19: 577273, 20: 69718, 21: 4213, 22: 1441046, 23: 4131447, 24: 1222335, 25: 152245, 26: 7497, 27: 901, 28: 198971, 29: 749244, 30: 593532, 31: 23881, 32: 1546929, 33: 577273, 34: 2634457, 35: 696127, 36: 38351, 37: 38054, 38: 13603}


### `run_copula_pipeline`

Pool construction -> keep/buy calibration -> either the swap sampler, or data augmentation ->
Kendall's $\tau$/hub score -> sparse C-vine -> sampling -> feasibility (transfer limit + budget)
-> squad deduplication, in one call. Returns `None` (with a printed reason) for gameweeks with
too little history, rather than raising, so the bulk loop below can skip them cleanly.


**Sparse vine.** Only players owned at least once in the history window enter the vine;
every other pool player is simulated as an independent uniform (their latent $u$ is pure
jitter, so there is nothing to fit). Pair-copulas are Gaussian-or-independent, selected
by BIC on one tree with $|\tau|<0.35$ forced to independence. The earlier setup (6
families, AIC, 15 trees, all ~120 players) was diagnosed at GW38/team 13603: its fit
was no better than a fit on column-permuted data (~480 spurious pairs both ways) and its
held-out log-likelihood was about $-900$ versus $0$ for independence. Each result now
records `null_loglik` and `holdout_loglik` so this check is visible every week.


**Swap sampler (default).** With `squad_sampler="swap"`, $\mathbb{W}_o$ is built as the
opponent's current squad plus keep/buy swaps (`generate_swap_squads`): each current
player is kept with the keep probability of eq. 3.6, and sold slots are refilled
like-for-like from non-owned players by $\text{softmax}(\tau_j\eta)$ without
replacement. Formation holds by construction; draws selling more than $T'$ players or
exceeding the current squad's value are rejected. The C-vine path is still available
with `squad_sampler="vine"`, but as diagnosed above it reduces to independence on this
data.


In [31]:
POSITIONS = ["Goalkeeper", "Defender", "Midfielder", "Forward"]
# Sparse vine settings. The original setup (6 families, AIC, 15 trees over all ~120 pool
# players) fit as many non-independent pairs on column-permuted data (dependence
# destroyed) as on the real data, and scored far below independence on held-out weeks.
# Gaussian-only + BIC + one tree + a |tau| threshold is the only configuration tried
# whose in-sample fit separates from that permutation null (see `diagnostics` below).
BICOP_FAMILY_SET = [pv.BicopFamily.indep, pv.BicopFamily.gaussian]
SELECTION_CRITERION = "bic"
TRUNC_LVL = 1
TAU_THRESHOLD = 0.35


def latent_uniforms(X_hist, P_hist, rng):
    """Distributional-transform latent uniforms u_{t,k} (see the note at the top of this notebook)."""
    a = np.where(X_hist.values == 0, 0.0, 1.0 - P_hist.values)
    b = np.where(X_hist.values == 0, 1.0 - P_hist.values, 1.0)
    draws = rng.uniform(size=X_hist.shape)
    U = a + draws * (b - a)
    return pd.DataFrame(U, index=X_hist.index, columns=X_hist.columns)


def kendall_cvine_order(U_vals):
    """C-vine root order (1-indexed) by descending hub score S_k = sum_j |tau_kj|."""
    m = U_vals.shape[1]
    tau_matrix = np.zeros((m, m))
    for i in range(m):
        for j in range(i + 1, m):
            tau, _ = kendalltau(U_vals[:, i], U_vals[:, j])
            tau = 0.0 if np.isnan(tau) else tau
            tau_matrix[i, j] = tau_matrix[j, i] = tau
    hub_scores = np.abs(tau_matrix).sum(axis=1)
    return (np.argsort(-hub_scores) + 1).tolist()


def fit_sparse_cvine(U_vals):
    cvine_structure = pv.CVineStructure(
        order=kendall_cvine_order(U_vals), trunc_lvl=TRUNC_LVL
    )
    cvine_controls = pv.FitControlsVinecop(
        family_set=BICOP_FAMILY_SET,
        selection_criterion=SELECTION_CRITERION,
        trunc_lvl=TRUNC_LVL,
        threshold=TAU_THRESHOLD,
    )
    return pv.Vinecop.from_data(
        U_vals, structure=cvine_structure, controls=cvine_controls
    )


def vine_diagnostics(U_vals, rng, holdout_weeks=(4, 6, 8)):
    """
    Checks whether the fitted dependence is real rather than noise:
    - null_loglik: in-sample log-lik of a vine refit on column-permuted U (dependence
      destroyed). The real fit's loglik should clearly exceed it.
    - holdout_loglik: mean log-lik on the last k weeks of a vine fit on the earlier
      ones. Independence scores exactly 0, so negative means the vine generalizes worse
      than independent sampling.
    """
    U_null = np.column_stack(
        [rng.permutation(U_vals[:, k]) for k in range(U_vals.shape[1])]
    )
    null_loglik = fit_sparse_cvine(U_null).loglik(U_null)
    holdout = [
        fit_sparse_cvine(U_vals[:-k]).loglik(U_vals[-k:])
        for k in holdout_weeks
        if U_vals.shape[0] - k >= 5
    ]
    holdout_loglik = float(np.mean(holdout)) if holdout else np.nan
    return null_loglik, holdout_loglik


def generate_swap_squads(
    team_id, GW, manager_fits, keep_buy_calib, T_LIMIT, n_samples, rng
):
    """
    W_o as "current squad plus up to T' swaps" (eq. 3.6's keep/buy process, sampled):
    the opponent's current squad is their round GW-1 squad; each player in it is kept
    with P(keep) = logit^-1(a + b*eta), and each position's sold slots are refilled
    like-for-like from the players they don't own, drawn by softmax(tau * eta) without
    replacement (Gumbel-top-k). eta = (beta^league + Delta_beta_m) . X on round GW's
    features (lagged, so known before the deadline). A current player missing from
    round GW's pool (no fixture in a blank gameweek) is scored on their round GW-1
    features instead; only one missing from both rounds is treated as having left the
    league and always sold.

    A draw is feasible if it sells at most T_LIMIT players and the new squad's value is
    at most the current squad's (no bank information, as elsewhere in this notebook).
    Formation is guaranteed by construction. Returns (squad_counts over feasible
    draws, stats dict).
    """
    value_map = (
        player_data.loc[player_data["round"] == GW].set_index("element")["value"].to_dict()
    )
    # prices as of the current squad's week, falling back to round GW for newcomers
    value_map.update(
        player_data.loc[player_data["round"] == GW - 1]
        .set_index("element")["value"]
        .to_dict()
    )

    n_sold = np.zeros(n_samples, dtype=int)
    squad_value = np.zeros(n_samples)
    current_value = 0.0
    per_position = []
    for position in POSITIONS:
        league_fit = get_league_fit_for_round(LEAGUE_FITS[position], GW)
        manager_fit = manager_fits[position]
        calib = keep_buy_calib[position]

        elements, X_raw = _position_pool(player_data, position, GW, league_fit["feats"])
        prev_elements, X_prev_raw = _position_pool(
            player_data, position, GW - 1, league_fit["feats"]
        )
        owned_map = get_owned(team_id, position, GW - 1, prev_elements)
        if owned_map is None:
            raise ValueError(
                f"no GW{GW - 1} {position} squad for team {team_id} to swap from"
            )
        owned = np.array([e for e, v in owned_map.items() if v == 1])
        beta_m = league_fit["beta_mean"] + manager_fit["delta_beta_mean"]
        eta = dict(
            zip(
                elements.tolist(),
                ((X_raw - league_fit["X_mean"]) / league_fit["X_std"]) @ beta_m,
            )
        )
        eta_prev = dict(
            zip(
                prev_elements.tolist(),
                ((X_prev_raw - league_fit["X_mean"]) / league_fit["X_std"]) @ beta_m,
            )
        )
        # blank-gameweek owners: fall back to last week's features (see docstring)
        eta_owned = [eta.get(e, eta_prev.get(e)) for e in owned]

        p_keep = np.array(
            [0.0 if h is None else expit(calib["a"] + calib["b"] * h) for h in eta_owned]
        )
        keep = rng.uniform(size=(n_samples, len(owned))) < p_keep[None, :]
        n_sold_pos = len(owned) - keep.sum(axis=1)

        owned_set = set(owned.tolist())
        candidates = np.array([e for e in elements if e not in owned_set])
        max_buy = int(n_sold_pos.max())
        if max_buy > 0:
            keys = calib["tau"] * np.array([eta[e] for e in candidates])[None, :]
            keys = keys + rng.gumbel(size=(n_samples, len(candidates)))
            top = np.argpartition(-keys, kth=max_buy - 1, axis=1)[:, :max_buy]
            # order the top max_buy by key, so the first n_sold_pos are the buys
            top = np.take_along_axis(
                top, np.argsort(-np.take_along_axis(keys, top, axis=1), axis=1), axis=1
            )
        else:
            top = np.zeros((n_samples, 0), dtype=int)
        bought = np.arange(max_buy)[None, :] < n_sold_pos[:, None]

        owned_values = np.array([value_map[e] for e in owned])
        candidate_values = np.array([value_map.get(e, np.nan) for e in candidates])
        squad_value += keep @ owned_values
        squad_value += np.where(bought, candidate_values[top], 0.0).sum(axis=1)
        current_value += owned_values.sum()
        n_sold += n_sold_pos
        per_position.append((owned, keep, candidates, top, bought))

    transfer_ok = n_sold <= T_LIMIT
    budget_ok = squad_value <= current_value
    feasible = np.where(transfer_ok & budget_ok)[0]

    squad_counts = {}
    for i in feasible:
        squad = []
        for owned, keep, candidates, top, bought in per_position:
            squad.extend(owned[keep[i]].tolist())
            squad.extend(candidates[top[i][bought[i]]].tolist())
        squad = tuple(sorted(squad))
        squad_counts[squad] = squad_counts.get(squad, 0) + 1

    stats = {
        "current_squad_value": current_value,
        "n_transfer_ok": int(transfer_ok.sum()),
        "n_budget_ok": int(budget_ok.sum()),
        "n_feasible": len(feasible),
        "n_transfers_distribution": np.bincount(n_sold).tolist(),
    }
    return squad_counts, stats


def run_copula_pipeline(
    GW,
    TEAM_ID,
    N_TOP=30,
    N_SAMPLES=20_000,
    T_LIMIT=None,
    min_history_weeks=1,
    seed=42,
    verbose=True,
    diagnostics=True,
    keep_buy=True,
    squad_sampler="swap",
):
    """
    `T_LIMIT=None` (the default) computes the manager's actual banked free transfers
    via `compute_free_transfers_available` instead of using a fixed constant -- pass an
    explicit int to override.

    `squad_sampler="swap"` (the default) builds W_o with generate_swap_squads: the
    opponent's current squad plus keep/buy swaps, which needs `keep_buy=True`.
    `squad_sampler="vine"` runs the C-vine sampling path below instead.
    """
    if squad_sampler not in ("swap", "vine"):
        raise ValueError(f"squad_sampler must be 'swap' or 'vine', got {squad_sampler!r}")
    if squad_sampler == "swap" and not keep_buy:
        raise ValueError("squad_sampler='swap' needs the keep/buy calibration (keep_buy=True)")
    if T_LIMIT is None:
        T_LIMIT = compute_free_transfers_available(TEAM_ID, GW, league_selections)

    rng = np.random.default_rng(seed)
    manager_fits = joblib.load(f"./estimates/manager_fits_{GW}.joblib")[GW]

    # keep/buy calibration (eq. 3.6) on the same history window as below
    keep_buy_calib = {
        position: fit_keep_buy_calibration(
            position, get_league_fit_for_round(LEAGUE_FITS[position], GW), GW - 1
        )
        if keep_buy
        else None
        for position in POSITIONS
    }

    # --- pool M ---
    owned_elements_ = []
    pool_records = []
    for position in POSITIONS:
        manager_fit = manager_fits[position]
        league_fit = get_league_fit_for_round(LEAGUE_FITS[position], GW)
        p_map, owned_map = position_marginals_at_week(
            TEAM_ID,
            position,
            manager_fit,
            league_fit,
            GW,
            rng,
            calib=keep_buy_calib[position],
        )

        ranked = sorted(p_map.items(), key=lambda kv: kv[1], reverse=True)
        top_elements = {e for e, _ in ranked[:N_TOP]}
        owned_elements = {e for e, v in (owned_map or {}).items() if v == 1}
        owned_elements_.append(owned_elements)
        for e in top_elements | owned_elements:
            pool_records.append({"element": e, "position": position})

    pool_df = (
        pd.DataFrame(pool_records)
        .sort_values(["position", "element"])
        .reset_index(drop=True)
    )
    elements_ordered = pool_df["element"].tolist()
    elements_arr = np.array(elements_ordered)
    position_arr = pool_df["position"].to_numpy()
    M = len(elements_ordered)

    # --- history + overhaul filtering ---
    all_history_rounds = sorted(
        league_selections.loc[
            (league_selections["team_id"] == TEAM_ID)
            & (
                league_selections["round"] < GW - 1
            ),  # & (league_selections['round'] >= 4),
            "round",
        ]
        .unique()
        .tolist()
    )

    # print('all_history_rounds: ', all_history_rounds)

    overhaul_rounds = [
        r
        for r in all_history_rounds
        if detect_squad_overhaul(TEAM_ID, r, league_selections)[1]
    ]
    history_rounds = [r for r in all_history_rounds if r not in overhaul_rounds]
    # print('history_rounds: ', history_rounds)
    if squad_sampler == "swap" and any(c is None for c in keep_buy_calib.values()):
        if verbose:
            print(
                f"GW{GW} team {TEAM_ID}: no league history before GW{GW - 1} to fit the "
                f"keep/buy calibration on, skipping"
            )
        return None

    if len(history_rounds) < min_history_weeks:
        if verbose:
            print(
                f"GW{GW} team {TEAM_ID}: only {len(history_rounds)} non-overhaul history weeks "
                f"(< {min_history_weeks}), skipping"
            )
        return None

    if squad_sampler == "swap":
        squad_counts, stats = generate_swap_squads(
            TEAM_ID, GW, manager_fits, keep_buy_calib, T_LIMIT, N_SAMPLES, rng
        )
        result = {
            "GW": GW,
            "team_id": TEAM_ID,
            "squad_sampler": squad_sampler,
            "M": M,
            "n_history_weeks": len(history_rounds),
            "overhaul_rounds": overhaul_rounds,
            "keep_buy_calib": keep_buy_calib,
            # vine-only diagnostics, kept so the summary table has the same columns
            "n_informative": np.nan,
            "vine_aic": np.nan,
            "vine_loglik": np.nan,
            "null_loglik": np.nan,
            "holdout_loglik": np.nan,
            "n_dep_pairs": np.nan,
            "n_samples": N_SAMPLES,
            "T_LIMIT": T_LIMIT,
            **stats,
            "n_unique_feasible_squads": len(squad_counts),
            "squad_counts": squad_counts,
        }
        if verbose:
            print(
                f"GW{GW:>2} team {TEAM_ID:>8}: swap sampler, T'={T_LIMIT}, "
                f"transfers per draw {stats['n_transfers_distribution']}, "
                f"feasible={stats['n_feasible']:>5}/{N_SAMPLES} ({len(squad_counts)} unique)"
            )
        return result

    # --- data augmentation ---
    X_hist = pd.DataFrame(index=history_rounds, columns=elements_ordered, dtype=float)
    P_hist = pd.DataFrame(index=history_rounds, columns=elements_ordered, dtype=float)
    for position in POSITIONS:
        manager_fit = manager_fits[position]
        league_fit = get_league_fit_for_round(LEAGUE_FITS[position], GW)
        pos_elements = pool_df.loc[pool_df["position"] == position, "element"].tolist()
        for rnd in history_rounds:
            p_map, owned_map = position_marginals_at_week(
                TEAM_ID,
                position,
                manager_fit,
                league_fit,
                rnd,
                rng,
                calib=keep_buy_calib[position],
            )
            for e in pos_elements:
                X_hist.loc[rnd, e] = (owned_map or {}).get(e, 0.0)
                P_hist.loc[rnd, e] = p_map.get(e, 1e-6)
    U_vals = latent_uniforms(X_hist, P_hist, rng).values

    # --- sparse C-vine over informative players only ---
    # A player never owned in the history window has u_{t,k} ~ Uniform[0, 1-p) every
    # week: pure jitter with no co-selection signal, so it's left out of the vine and
    # simulated independently below.
    informative_idx = np.where(X_hist.values.sum(axis=0) > 0)[0]
    vine, null_loglik, holdout_loglik = None, np.nan, np.nan
    n_dep_pairs, aic, vine_loglik = 0, 0.0, 0.0
    if len(informative_idx) >= 2:
        U_info = U_vals[:, informative_idx]
        vine = fit_sparse_cvine(U_info)
        n_dep_pairs = sum(
            f != pv.BicopFamily.indep for tree in vine.families for f in tree
        )
        aic = vine.aic(U_info)
        vine_loglik = vine.loglik(U_info)
        if diagnostics:
            null_loglik, holdout_loglik = vine_diagnostics(U_info, rng)

    # --- target-week alpha / ownership / value ---
    alpha_target, current_owned = {}, {}
    for position in POSITIONS:
        manager_fit = manager_fits[position]
        league_fit = get_league_fit_for_round(LEAGUE_FITS[position], GW)
        alpha_map, owned_map = position_alpha_at_week(
            TEAM_ID, position, manager_fit, league_fit, GW
        )
        for e in pool_df.loc[pool_df["position"] == position, "element"]:
            alpha_target[e] = alpha_map.get(e, 1e-12)
            current_owned[e] = (owned_map or {}).get(e, 0.0)

    alpha_vec = np.array([alpha_target[e] for e in elements_ordered])
    current_vec = np.array([current_owned[e] for e in elements_ordered])
    position_arr = pool_df["position"].to_numpy()

    value_map = (
        player_data.loc[player_data["round"] == GW - 1]
        .set_index("element")["value"]
        .to_dict()
    )
    value_vec = np.array([value_map.get(e, np.nan) for e in elements_ordered])
    current_squad_value = float(current_vec @ value_vec)

    # --- sampling + constrained per-position selection ---
    sim_U = rng.uniform(size=(N_SAMPLES, M))
    if vine is not None:
        sim_U[:, informative_idx] = vine.simulate(n=N_SAMPLES, seeds=[seed])
    w_star = np.zeros_like(sim_U, dtype=int)
    for position, k in SQUAD_SLOTS.items():
        idxs = np.where(position_arr == position)[0]
        w = np.clip(alpha_vec[idxs], 1e-12, None)
        keys = sim_U[:, idxs] ** (1.0 / w)[None, :]
        topk = np.argpartition(-keys, kth=k - 1, axis=1)[:, :k]
        rows = np.repeat(np.arange(sim_U.shape[0]), k)
        cols = idxs[topk.ravel()]
        w_star[rows, cols] = 1

    transfers = 0.5 * np.abs(w_star - current_vec[None, :]).sum(axis=1)
    squad_values = w_star @ value_vec
    transfer_ok = transfers <= T_LIMIT
    budget_ok = squad_values <= current_squad_value
    feasible_mask = transfer_ok & budget_ok

    feasible_squads = [
        tuple(elements_arr[row.astype(bool)].tolist()) for row in w_star[feasible_mask]
    ]
    squad_counts = {}
    for sq in feasible_squads:
        squad_counts[sq] = squad_counts.get(sq, 0) + 1

    result = {
        "GW": GW,
        "team_id": TEAM_ID,
        "squad_sampler": squad_sampler,
        "M": M,
        "n_history_weeks": len(history_rounds),
        "overhaul_rounds": overhaul_rounds,
        "keep_buy_calib": keep_buy_calib,
        "n_informative": len(informative_idx),
        "vine_aic": aic,
        "vine_loglik": vine_loglik,
        "null_loglik": null_loglik,
        "holdout_loglik": holdout_loglik,
        "n_dep_pairs": n_dep_pairs,
        "current_squad_value": current_squad_value,
        "n_samples": N_SAMPLES,
        "T_LIMIT": T_LIMIT,
        "n_transfer_ok": int(transfer_ok.sum()),
        "n_budget_ok": int(budget_ok.sum()),
        "n_feasible": int(feasible_mask.sum()),
        "n_unique_feasible_squads": len(squad_counts),
        "squad_counts": squad_counts,
    }
    if verbose:
        print(
            f"GW{GW:>2} team {TEAM_ID:>8}: M={M:>3}, history={len(history_rounds):>2}wks "
            f"({len(overhaul_rounds)} overhaul), T'={T_LIMIT}, vine dims={len(informative_idx)}, "
            f"dep pairs={n_dep_pairs}, LL={vine_loglik:.1f} (null {null_loglik:.1f}, "
            f"holdout {holdout_loglik:.2f}), "
            f"feasible={result['n_feasible']:>5}/{N_SAMPLES} ({result['n_unique_feasible_squads']} unique)"
        )
    return result

### Worked example: swap vs vine

Both samplers for the gameweek and opponent set at the top. The vine result carries its
diagnostics: `vine_loglik` should clearly exceed `null_loglik` (a refit with the dependence
shuffled out), and `holdout_loglik` (later weeks scored by a vine fit on earlier ones) should be
above 0, the score of independent sampling.


In [ ]:
example = {
    sampler: run_copula_pipeline(GW, TEAM_ID, N_SAMPLES=20_000, seed=42, squad_sampler=sampler)
    for sampler in ["swap", "vine"]
}
pd.DataFrame(
    {
        sampler: {
            k: v
            for k, v in res.items()
            if k not in ("squad_counts", "keep_buy_calib", "overhaul_rounds")
        }
        for sampler, res in example.items()
        if res is not None
    }
)


### Bulk run

Loop over every `GW` in 6-38 with `opponents_by_round[GW]` as that week's team, exactly
as `longitudinal.ipynb`'s calibration sweep did. `N_SAMPLES` is 20k to keep 33 runs
tractable; raise it per gameweek (`run_copula_pipeline(gw, team_id, N_SAMPLES=...)`) for a
more precise estimate. Results are saved to
`./estimates/copula_results_all.joblib`, keyed by `GW`, mirroring
`manager_fits_{GW}.joblib`'s convention.


In [34]:
results_by_gw = {}
for gw in range(6, 39):
    team_id = opponents_by_round[gw]
    try:
        results_by_gw[gw] = run_copula_pipeline(
            gw, team_id, N_SAMPLES=20_000, seed=42
        )  # T_LIMIT: dynamic (banked FTs)
    except Exception as e:
        print(f"GW{gw} team {team_id} FAILED: {e}")
        results_by_gw[gw] = None

joblib.dump(results_by_gw, "./estimates/copula_results_all.joblib")

n_ok = sum(1 for r in results_by_gw.values() if r is not None)
print(f"\n{n_ok} / {len(results_by_gw)} gameweeks produced a result")

GW21 team     4213: M=120, history=18wks (1 overhaul), T'=1, AIC=   -984.9, feasible=   40/20000 (24 unique)
GW22 team  1441046: M=120, history=19wks (1 overhaul), T'=1, AIC=  -1098.9, feasible=  123/20000 (71 unique)
GW23 team  4131447: M=120, history=20wks (1 overhaul), T'=5, AIC=   -965.9, feasible=10852/20000 (10460 unique)
GW24 team  1222335: M=120, history=21wks (1 overhaul), T'=2, AIC=   -951.9, feasible= 1942/20000 (1322 unique)
GW25 team   152245: M=120, history=22wks (1 overhaul), T'=1, AIC=   -995.3, feasible=  235/20000 (104 unique)
GW26 team     7497: M=121, history=23wks (1 overhaul), T'=2, AIC=  -1029.7, feasible=  501/20000 (380 unique)
GW27 team      901: M=121, history=24wks (1 overhaul), T'=1, AIC=  -1023.2, feasible=   19/20000 (10 unique)
GW28 team   198971: M=120, history=26wks (0 overhaul), T'=2, AIC=   -904.2, feasible= 1340/20000 (921 unique)
GW29 team   749244: M=120, history=26wks (1 overhaul), T'=1, AIC=   -925.8, feasible=   51/20000 (23 unique)
GW30 team  

In [35]:
# Tidy summary across the season. Early gameweeks (GW6-7, <5 non-overhaul history
# weeks) are skipped entirely (result is None); some later ones (e.g. GW10, if the
# manager's history is mostly overhaul weeks) can still run but land on a degenerate
# independence-only vine (AIC=0.0) with 0 feasible samples -- both are visible below
# rather than hidden. T_LIMIT is each week's dynamically-computed banked free
# transfers, not a fixed constant (see compute_free_transfers_available).
summary_rows = []
for gw, res in results_by_gw.items():
    if res is None:
        summary_rows.append(
            {
                "GW": gw,
                "team_id": opponents_by_round[gw],
                "status": "skipped (too little history)",
            }
        )
    else:
        summary_rows.append(
            {
                "GW": gw,
                "team_id": res["team_id"],
                "status": "ok",
                "n_history_weeks": res["n_history_weeks"],
                "T_LIMIT": res["T_LIMIT"],
                "n_informative": res["n_informative"],
                "n_dep_pairs": res["n_dep_pairs"],
                "vine_loglik": res["vine_loglik"],
                "null_loglik": res["null_loglik"],
                "holdout_loglik": res["holdout_loglik"],
                "n_feasible": res["n_feasible"],
                "n_unique_feasible_squads": res["n_unique_feasible_squads"],
                "current_squad_value": res["current_squad_value"],
            }
        )

summary_df = pd.DataFrame(summary_rows)
summary_df

,GW,team_id,status,n_history_weeks,T_LIMIT,vine_aic,n_feasible,n_unique_feasible_squads,current_squad_value
0,21,4213,ok,18,1,-984.932073,40,24,1046.0
1,22,1441046,ok,19,1,-1098.910408,123,71,1042.0
2,23,4131447,ok,20,5,-965.898512,10852,10460,1016.0
3,24,1222335,ok,21,2,-951.941715,1942,1322,1042.0
4,25,152245,ok,22,1,-995.299662,235,104,1046.0
5,26,7497,ok,23,2,-1029.707698,501,380,1013.0
6,27,901,ok,24,1,-1023.172179,19,10,1022.0
7,28,198971,ok,26,2,-904.236443,1340,921,1023.0
8,29,749244,ok,26,1,-925.842050,51,23,1012.0
9,30,593532,ok,27,3,-929.522088,0,0,810.0
